# 三项冻结验证的结果复核
只读取随包表格，不加载模型、不拟合读出、不访问网络。正式事实源为同目录 README 与验证脚本。

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
root = Path.cwd()
assert (root / 'event_targets.csv').exists(), '请在本笔记本目录运行'
events = pd.read_csv(root / 'event_targets.csv')
clare = pd.read_csv(root / 'clare_targets.csv')
assert events.target_matches_float32.all()
assert ((clare.score >= 5).astype(int) == clare.standard_class).all()
print('事件目标一致:', len(events), 'CLARE 窗口:', len(clare))


事件目标一致: 1462 CLARE 窗口: 794


## 匹配非事件与时间扰动
单位为微西门子，以下按受试者等权；描述性比较不构成因果效应。

In [2]:
subjects = pd.read_csv(root / 'event_subjects.csv')
stability = pd.read_csv(root / 'event_stability.csv')
print(subjects[['subject','paired_signed_difference','paired_absolute_difference']].to_string(index=False))
print(stability.groupby('comparison')[['change_rmse','spearman','sign_flip_fraction']].mean().round(5).to_string())


  subject  paired_signed_difference  paired_absolute_difference
sub-cp003                 -0.049298                    0.019117
sub-cp004                 -0.023301                    0.035857
sub-cp008                 -0.072006                   -0.042494
sub-cp011                 -0.021168                    0.042305
sub-cp019                  0.010624                   -0.004506
sub-cp022                  0.005001                    0.019450
sub-cp025                  0.003658                    0.001375
             change_rmse  spearman  sign_flip_fraction
comparison                                            
first_half       0.13570   0.87158             0.09397
second_half      0.14710   0.92816             0.08575
shift_-0.5       0.07883   0.92964             0.04815
shift_-1.0       0.14073   0.81655             0.10084
shift_0.5        0.07701   0.94281             0.05336
shift_1.0        0.14000   0.84715             0.10251


## 非零时差与互补性进入判据
时间误差是各条件均方根误差的等权均值；稳定门是开发筛选规则，未宣称统计显著。

In [3]:
timing = pd.read_csv(root / 'timing_aggregates.csv')
q = timing[(timing.condition_group == '非零时钟偏移') & (timing.method == 'chronaris') & (timing.variant == 'reference')]
print(q.pivot(index=['route','target'],columns='arm',values='rmse').round(5).to_string())
gates = pd.read_csv(root / 'complementarity_gates.csv')
main = gates[gates.primary_task & ~gates.exploratory]
assert len(main) == 24
print('标准主要对照:', len(main), '稳定联合通过:', int(main.joint_stable_pass.sum()))


arm                                 model  train_mean  train_median      zero
route           target                                                       
self_supervised clock_magnitude   1.16345     1.01389       1.29167   1.41667
                response_lag     17.51945     9.60494       9.12357  15.92216
                signed_clock      1.41785     1.41667       1.41667   1.41667
task_guided     clock_magnitude   0.96235     1.01389       1.29167   1.41667
                response_lag     19.56182     9.60494       9.12357  15.92216
                signed_clock      1.41870     1.41667       1.41667   1.41667
标准主要对照: 24 稳定联合通过: 0


本笔记本各代码单元已用相同 Python 解释器按顺序执行并保存文本输出。环境未安装 Jupyter 内核，未做 notebook 图形界面渲染检查；表格以同目录 Markdown 为主要阅读入口。